# 00 Normalizar IDS : decomisos individuales, Europa, 2011 a 30/06/2026

Unir los 7 Excel de IDS, limpiarlos y dejar una tabla de detalle (1 fila = 1 decomiso).



In [27]:
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
try:
    display
except NameError:
    display = print

RAIZ = Path.cwd().parent if Path.cwd().name == "01_notebooks" else Path.cwd()
FUENTES = RAIZ / "00_fuentes" / "unodc_ids_decomisos_individuales"
SILVER = RAIZ / "02_datos" / "silver"
SILVER.mkdir(parents=True, exist_ok=True)

PATRONES_EXCEL = ["unodc_ids_*.xlsx", "ONUdrugs*.xlsx"]

assert FUENTES.exists(), "No existe la carpeta de IDS en 00_fuentes"
for patron in PATRONES_EXCEL:
    for f in sorted(FUENTES.glob(patron)):
        print(f.name)

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

ONUdrugs _2011-2016.xlsx
ONUdrugs_2017-2019.xlsx
ONUdrugs_2020 (Ene-Dic).xlsx
ONUdrugs_2021 (Ene-Dic).xlsx
ONUdrugs_2022 (Ene-Dic).xlsx
ONUdrugs_2023 (Ene-Dic).xlsx
ONUdrugs_2024-2026 (30_06).xlsx


## Registro de la limpieza
Función para registrar cada paso. 
Guardado como:  02_datos/silver/log_ids_limpieza.csv.

In [28]:
registro = []

def registrar(paso, que_se_hizo, antes, despues, afectadas=0):
    registro.append({"paso": paso, "que_se_hizo": que_se_hizo,
                     "filas_antes": antes, "filas_despues": despues,
                     "filas_afectadas": afectadas})
    print(f"Paso {paso}: {que_se_hizo}\n   filas antes={antes:,} | después={despues:,} | afectadas={afectadas:,}")

## Carga de los Excel


In [29]:
ficheros = sorted({f for patron in PATRONES_EXCEL for f in FUENTES.glob(patron)})
assert ficheros, "No hay Excel de IDS en " + str(FUENTES)
print(f"{len(ficheros)} ficheros encontrados")

partes = []
for fichero in ficheros:
    tabla = pd.read_excel(fichero, dtype=str)
    tabla["archivo_origen"] = fichero.name
    partes.append(tabla)

df = pd.concat(partes, ignore_index=True)
registrar(1, f"Carga de {len(ficheros)} ficheros", 0, len(df), len(ficheros))
display(df.groupby("archivo_origen").size().rename("filas"))

7 ficheros encontrados
Paso 1: Carga de 7 ficheros
   filas antes=0 | después=718,856 | afectadas=7


archivo_origen
ONUdrugs _2011-2016.xlsx           136651
ONUdrugs_2017-2019.xlsx             92159
ONUdrugs_2020 (Ene-Dic).xlsx        37312
ONUdrugs_2021 (Ene-Dic).xlsx       117164
ONUdrugs_2022 (Ene-Dic).xlsx        80909
ONUdrugs_2023 (Ene-Dic).xlsx       141625
ONUdrugs_2024-2026 (30_06).xlsx    113036
Name: filas, dtype: int64

In [30]:
display(df.head(3))
df.info()

,Row Number,Seizure Date,Country/Territory of Seizure,ISO3 Code,Region,Subregion,City,Administrative Region,Drug/Substance,Measurement Unit,Quantity Seized,Physical Seizure Location,Trafficking Mode of Transportation,archivo_origen
0,977397,2016-12-31 00:00:00,Spain,ESP,Europe,Western and Central Europe,Granada,Granada,Cannabis herb (marijuana),g,38.9,Other outdoor/open air sites/facilities,Other,ONUdrugs _2011-2016.xlsx
1,977396,2016-12-31 00:00:00,Spain,ESP,Europe,Western and Central Europe,Tarifa,Cádiz,Cannabis resin (hashish),g,35000,Data not available,Other,ONUdrugs _2011-2016.xlsx
2,977395,2016-12-31 00:00:00,Spain,ESP,Europe,Western and Central Europe,Granada,Granada,Cannabis resin (hashish),g,10,Other outdoor/open air sites/facilities,Other,ONUdrugs _2011-2016.xlsx


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 718856 entries, 0 to 718855
Data columns (total 14 columns):
 #   Column                              Non-Null Count   Dtype 
---  ------                              --------------   ----- 
 0   Row Number                          718849 non-null  object
 1   Seizure Date                        718842 non-null  object
 2   Country/Territory of Seizure        718842 non-null  object
 3   ISO3 Code                           718842 non-null  object
 4   Region                              718842 non-null  object
 5   Subregion                           718842 non-null  object
 6   City                                696174 non-null  object
 7   Administrative Region               685295 non-null  object
 8   Drug/Substance                      718842 non-null  object
 9   Measurement Unit                    718842 non-null  object
 10  Quantity Seized                     718842 non-null  object
 11  Physical Seizure Location           718

## Quitar las filas de pie
Filas vacías sin row number reconocible. 


In [31]:
es_decomiso = pd.to_numeric(df["Row Number"], errors="coerce").notna()
pies = df.loc[~es_decomiso, "Row Number"].dropna().str.replace("\n", " ").unique()
display(pies)

antes = len(df)
df = df[es_decomiso].copy()
registrar(2, "Quitar filas de pie (vacías y 'Filtros aplicados')", antes, len(df), antes - len(df))

array(['Filtros aplicados:  seizure_date el o después del 01/01/2011 y antes del 01/01/2017 region es Europe',
       'Filtros aplicados:  seizure_date el o después del 01/01/2017 y antes del 01/01/2020 region es Europe',
       'Filtros aplicados:  seizure_date el o después del 01/01/2020 y antes del 01/01/2021 region es Europe',
       'Filtros aplicados:  seizure_date el o después del 01/01/2021 y antes del 01/01/2022 region es Europe',
       'Filtros aplicados:  seizure_date el o después del 01/01/2022 y antes del 01/01/2023 region es Europe',
       'Filtros aplicados:  seizure_date el o después del 01/01/2023 y antes del 01/01/2024 region es Europe',
       'Filtros aplicados:  seizure_date el o después del 01/01/2024 y antes del 01/07/2026 region es Europe'],
      dtype=object)

Paso 2: Quitar filas de pie (vacías y 'Filtros aplicados')
   filas antes=718,856 | después=718,842 | afectadas=14


## Duplicados por id
Row Number, el identificador de UNODC.

In [32]:
df["Row Number"] = df["Row Number"].astype("int64")
antes = len(df)
n_repetidos = df["Row Number"].duplicated().sum()
df = df.drop_duplicates(subset="Row Number", keep="first")
registrar(3, "Duplicados por Row Number", antes, len(df), n_repetidos)

Paso 3: Duplicados por Row Number
   filas antes=718,842 | después=718,842 | afectadas=0


## Nombres de columna
Nombres en español y en snake_case.

In [33]:
df = df.rename(columns={
    "Row Number": "id_decomiso",
    "Seizure Date": "fecha",
    "Country/Territory of Seizure": "pais",
    "ISO3 Code": "iso3",
    "Region": "region",
    "Subregion": "subregion_original",
    "City": "ciudad_original",
    "Administrative Region": "region_administrativa",
    "Drug/Substance": "droga_original",
    "Measurement Unit": "unidad_original",
    "Quantity Seized": "cantidad_original",
    "Physical Seizure Location": "lugar_fisico",
    "Trafficking Mode of Transportation": "transporte",
})
registrar(4, "Renombrar columnas", len(df), len(df))
display(df.columns.tolist())

Paso 4: Renombrar columnas
   filas antes=718,842 | después=718,842 | afectadas=0


['id_decomiso',
 'fecha',
 'pais',
 'iso3',
 'region',
 'subregion_original',
 'ciudad_original',
 'region_administrativa',
 'droga_original',
 'unidad_original',
 'cantidad_original',
 'lugar_fisico',
 'transporte',
 'archivo_origen']

## Tipos de dato
fecha a tipo fecha y cantidad_original a número. Con errors="coerce" lo que no se pueda convertir
se convierte en NaN, recuento.

In [34]:
df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce").dt.normalize()
df["cantidad_original"] = pd.to_numeric(df["cantidad_original"], errors="coerce")

fechas_mal = df["fecha"].isna().sum()
cantidades_mal = df["cantidad_original"].isna().sum()
registrar(5, f"Tipos: fechas no válidas={fechas_mal}, cantidades no numéricas={cantidades_mal}. "
             f"Rango {df.fecha.min().date()} a {df.fecha.max().date()}",
          len(df), len(df), fechas_mal + cantidades_mal)

Paso 5: Tipos: fechas no válidas=0, cantidades no numéricas=0. Rango 2011-01-01 a 2026-06-30
   filas antes=718,842 | después=718,842 | afectadas=0


## Espacios sobrantes
Espacios al principio, al final o dobles. Evita que "Spain" y "Spain " parezcan países distintos.

In [35]:
cols_texto = ["pais", "iso3", "region", "subregion_original", "ciudad_original", "region_administrativa",
              "droga_original", "unidad_original", "lugar_fisico", "transporte"]

tocadas = 0
for col in cols_texto:
    limpio = df[col].str.replace(r"\s+", " ", regex=True).str.strip()
    tocadas += (df[col].notna() & (df[col] != limpio)).sum()
    df[col] = limpio
registrar(6, "Quitar espacios sobrantes en columnas de texto", len(df), len(df), tocadas)

Paso 6: Quitar espacios sobrantes en columnas de texto
   filas antes=718,842 | después=718,842 | afectadas=2,210


## Nulos
Primero se miran los nulos reales. Después:
- ciudad y region_administrativa vacías a "Sin dato".
- "Data not available" en lugar y transporte a "No disponible".
- Se crean dos columnas booleanas para poder filtrar solo lo que sí tiene dato.

In [36]:
nulos = df.isna().sum()
display(nulos[nulos > 0])

ciudad_original          22668
region_administrativa    33547
dtype: int64

In [37]:
df[["ciudad_original", "region_administrativa"]] = df[["ciudad_original", "region_administrativa"]].fillna("Sin dato")
for col in ["lugar_fisico", "transporte"]:
    df[col] = df[col].replace({"Data not available": "No disponible"}).fillna("Sin dato")

df["lugar_disponible"] = ~df["lugar_fisico"].isin(["No disponible", "Sin dato"])
df["transporte_disponible"] = ~df["transporte"].isin(["No disponible", "Sin dato"])

registrar(7, f"Nulos: {nulos[nulos > 0].to_dict()}. 'Data not available' a 'No disponible'",
          len(df), len(df), (~df.lugar_disponible).sum() + (~df.transporte_disponible).sum())
print(f"Con lugar informado: {df.lugar_disponible.mean():.0%} | con transporte informado: {df.transporte_disponible.mean():.0%}")

Paso 7: Nulos: {'ciudad_original': 22668, 'region_administrativa': 33547}. 'Data not available' a 'No disponible'
   filas antes=718,842 | después=718,842 | afectadas=1,094,512
Con lugar informado: 31% | con transporte informado: 17%


## País
Un mismo país puede aparecer con dos nombres. El código iso3 es la clave fiable: se usa un único nombre por iso3.

In [38]:
print("Países antes:", df["pais"].nunique())
display(df.groupby("iso3")["pais"].nunique().loc[lambda s: s > 1])      

nombre_por_iso3 = df.groupby("iso3")["pais"].agg(lambda s: s.value_counts().index[0])
cambiadas = (df["pais"] != df["iso3"].map(nombre_por_iso3)).sum()
df["pais"] = df["iso3"].map(nombre_por_iso3)

print("Países después:", df["pais"].nunique())
registrar(8, "Un solo nombre por iso3 (ej. 'Netherlands (Kingdom of the)' a 'Netherlands')", len(df), len(df), cambiadas)

Países antes: 44


iso3
NLD    2
Name: pais, dtype: int64

Países después: 43
Paso 8: Un solo nombre por iso3 (ej. 'Netherlands (Kingdom of the)' a 'Netherlands')
   filas antes=718,842 | después=718,842 | afectadas=5


## Subregión
UNODC cambió los nombres de las subregiones entre entregas. Se unifican con un diccionario.

In [39]:
display(df["subregion_original"].value_counts())

subregion_original
West and Central Europe       475605
Western and Central Europe    122467
East Europe                    55490
Eastern Europe                 47649
Southeast Europe               15414
South-Eastern Europe            2217
Name: count, dtype: int64

In [40]:
MAPA_SUBREGION = {
    "Western and Central Europe": "West and Central Europe",
    "Eastern Europe": "East Europe",
    "South-Eastern Europe": "Southeast Europe",
}
df["subregion"] = df["subregion_original"].replace(MAPA_SUBREGION)

cambiadas = (df["subregion"] != df["subregion_original"]).sum()
con_varias = (df.groupby("iso3")["subregion"].nunique() > 1).sum()
registrar(9, f"Unificar subregiones. Países con más de una subregión después: {con_varias}", len(df), len(df), cambiadas)
display(df["subregion"].value_counts())

Paso 9: Unificar subregiones. Países con más de una subregión después: 0
   filas antes=718,842 | después=718,842 | afectadas=172,333


subregion
West and Central Europe    598072
East Europe                103139
Southeast Europe            17631
Name: count, dtype: int64

## Droga: limpiar el nombre y agrupar
- Limpiar el nombre: quitar espacios alrededor de guiones (JWH - 018 a JWH-018) y juntar
variantes de mayúsculas o erratas conocidas.
-Agrupar: hay 262 sustancias. Las agrupamos en pocas familias con palabras clave (GRUPOS).
La clasificación es propia, no oficial de UNODC.
El orden del diccionario importa: gana la primera familia que encaja.

In [41]:
df["droga"] = (df["droga_original"]
               .str.replace(r"\s*-\s*", "-", regex=True)
               .replace({"Flunitrazepam (Disomnilan, Flumipam, Rophynol)":
                         "Flunitrazepam (Disomnilan, Flumipam, Rohypnol)"}))

mas_frecuente = df.groupby(df["droga"].str.casefold())["droga"].agg(lambda s: s.value_counts().index[0])
df["droga"] = df["droga"].str.casefold().map(mas_frecuente)

print("Nombres de droga antes:", df["droga_original"].nunique(), "| después:", df["droga"].nunique())

Nombres de droga antes: 262 | después: 261


In [42]:
GRUPOS = {
    "Cannabinoides sinteticos": r"synthetic cannabinoid|spice|jwh|hu-210|cp 47|cp-55|akb48|pb-22|indazole|indole|carboxamy?de|carboxylate|xlr-11|thj|adb-|bb-22|fub|rcs-4|mmb|mdmb|sdb-006|nm-2201|naphth|quinoline-8|adamant|methanone|1 benzyl|3-butenoyl|chmin|adbica",
    "Cannabis": r"cannabis|hashish|marijuana|tetrahydrocannabinol|\bthc\b",
    "Cocaina": r"cocaine|crack|\bcoca\b",
    "Opioides": r"heroin|opium|morphine|poppy|methadone|buprenorphine|subutex|oxycodone|codeine|tramadol|fentanyl|carfentanil|desomorphine|opioid|opiate|promedol|substitol|butorphanol|tilidate|dextropoxyphene|nitazene|6-mono|6-desoxy|codidol|mppp|levomethorphan|phentanyl",
    "Catinonas sinteticas y khat": r"cathinon|cathinone|mephedrone|ephedrone|\bcmc\b|-cmc|pvp|mdpv|methylone|ethylone|hexedrone|khat|\bmmc\b|bk_mdea|keto|clephedrone",
    "Estimulantes tipo anfetamina y extasis": r"amphetamine|\bmdma\b|ecstasy|\bmda\b|mdea|mmda|mbdb|captagon|fenfluramine|methylhexanamine|dimethylamphetamine|mapb|fluoromethamphetamine",
    "Benzodiacepinas y sedantes": r"benzodiazepine|alprazolam|clonazepam|diazepam|lorazepam|oxazepam|flunitrazepam|brotizolam|clorazepate|tofisopam|zolpidem|barbiturate|phenobarbital|sedative|\bghb\b|\bgbl\b|pregabalin|methaqualone|phenibut|somnubene",
    "Disociativos": r"ketamine|phencyclidine|eticyclidine",
    "Alucinogenos": r"\blsd\b|psilocybin|dimethyltryptamine|tryptamine|2c-|hallucinogen|\bdob\b|\bstp\b|salvinorin|5-meo|4-hydroxytryptamine",
    "Precursores y quimicos": r"precursor|acid\b|acetone|anhydride|ether|methanol|toluene|p2np|nitropropene|\bbmk\b|\bpmk\b|apaan|apaa\b|phenylacetone|mapa\b|nitroethane|methylamine|sodium carbonate|permanganate|pyrrolidine|solvent|mdp2p|adulterant|tartaric|methylester|quinoline|ephedrine|pseudoephedrine",
    "Farmacos y esteroides": r"nandrolone|steroid|modafinil|amantadine|tianeptine|trihexiphenidyl|amfepramone|pharmaceutical|dextromethorphan|pipradrol",
    "Otras sustancias psicoactivas nuevas (NPS)": r"\bnps\b|piperazine|\bbzp\b|mcpp|tfmpp|phenethylamine|aminoindane|kratom|mitragyn|methiopropamine|thiopropamine|dimethocaine|4-ci",
}

def clasificar(nombre):
    for grupo, patron in GRUPOS.items():
        if re.search(patron, nombre, flags=re.IGNORECASE):
            return grupo
    return "Otras / sin clasificar"

dim_droga = pd.DataFrame({"droga": df["droga"].unique()})
dim_droga["grupo_droga"] = dim_droga["droga"].map(clasificar)
dim_droga = dim_droga.sort_values(["grupo_droga", "droga"]).reset_index(drop=True)

df["grupo_droga"] = df["droga"].map(dim_droga.set_index("droga")["grupo_droga"])

registrar(10, f"Droga: {df.droga_original.nunique()} a {df.droga.nunique()} nombres, {df.grupo_droga.nunique()} grupos "
              f"(clasificación propia)", len(df), len(df), (df.droga != df.droga_original).sum())
display(df["grupo_droga"].value_counts())

Paso 10: Droga: 262 a 261 nombres, 13 grupos (clasificación propia)
   filas antes=718,842 | después=718,842 | afectadas=2,778


grupo_droga
Cannabis                                      351193
Cocaina                                       118870
Opioides                                       60367
Estimulantes tipo anfetamina y extasis         57542
Catinonas sinteticas y khat                    37209
Benzodiacepinas y sedantes                     35275
Cannabinoides sinteticos                       25724
Alucinogenos                                   11888
Otras / sin clasificar                         10871
Disociativos                                    3951
Otras sustancias psicoactivas nuevas (NPS)      3668
Precursores y quimicos                          2150
Farmacos y esteroides                            134
Name: count, dtype: int64

In [43]:
display(df.loc[df.grupo_droga == "Otras / sin clasificar", "droga"].value_counts().head(15))

droga
Other-miscellaneous                         5416
Other non-specified drugs/substances        4375
Non-specified synthetic drugs/substances     278
Non-specified psychotropic substances        249
Taren                                        221
Multiple drugs                               123
Non-specified stimulants                     106
Other drugs/substances                        97
Synthetics                                     3
Controlled Drug Mix                            3
Name: count, dtype: int64

## Equivalencia de grupos de droga con el Annex 7.1
IDS y 7.1  clasifican las drogas de forma distinta. 
Para poder compararlas en Power BI hace falta una columna común: grupo_unodc_71, con el mismo nombre que usa 7.1.
La equivalencia es propia y parcial: solo se enlazan los grupos cuya correspondencia es clara.
El resto queda como "Sin equivalente directo". Para este proyecto importan: cocaína, cannabis y opioides.

In [44]:
EQUIVALENCIA_71 = {
    "Cocaina": "Cocaine-type",
    "Cannabis": "Cannabis-type drugs (excluding synthetic cannabinoids)",
    "Opioides": "Opioids",
    "Benzodiacepinas y sedantes": "Sedatives and tranquillizers",
    "Alucinogenos": "Hallucinogens",
    "Precursores y quimicos": "Precursors",
    "Otras sustancias psicoactivas nuevas (NPS)": "NPS",
}
df["grupo_unodc_71"] = df["grupo_droga"].map(EQUIVALENCIA_71).fillna("Sin equivalente directo")

equivalencia = (df[["grupo_droga", "grupo_unodc_71"]].drop_duplicates()
                .sort_values("grupo_droga").reset_index(drop=True))
registrar(11, "Equivalencia de grupos IDS a 7.1 (parcial)", len(df), len(df),
          (df.grupo_unodc_71 != "Sin equivalente directo").sum())
display(equivalencia)

Paso 11: Equivalencia de grupos IDS a 7.1 (parcial)
   filas antes=718,842 | después=718,842 | afectadas=583,411


,grupo_droga,grupo_unodc_71
0,Alucinogenos,Hallucinogens
1,Benzodiacepinas y sedantes,Sedatives and tranquillizers
2,Cannabinoides sinteticos,Sin equivalente directo
3,Cannabis,Cannabis-type drugs (excluding synthetic canna...
4,Catinonas sinteticas y khat,Sin equivalente directo
5,Cocaina,Cocaine-type
6,Disociativos,Sin equivalente directo
7,Estimulantes tipo anfetamina y extasis,Sin equivalente directo
8,Farmacos y esteroides,Sin equivalente directo
9,Opioides,Opioids


## Unidades y cantidades
Las cantidades vienen en unidades distintas (g, kg, litros, pastillas, plantas…). No se pueden sumar juntas.
- g y kg a cantidad_kg (g dividido entre 1000)
- lt a cantidad_litros
- pastillas, unidades, plantas a cantidad_unidades
- Se conserva cantidad_original y unidad_original.
- Cantidades menores o iguales a 0 se marcan (cantidad_no_positiva) y no se convierten.

In [45]:
display(df["unidad_original"].value_counts())

unidad_original
g              419572
kg             205213
tablet/pill     60453
unit            15947
plant           11919
lt               5721
other              14
Ampoule             3
Name: count, dtype: int64

In [46]:
TIPO_MEDIDA = {"g": "masa", "kg": "masa", "lt": "volumen",
               "tablet/pill": "unidades", "unit": "unidades", "Ampoule": "unidades",
               "plant": "plantas", "other": "otro"}
FACTOR_A_KG = {"kg": 1, "g": 0.001}

df["tipo_medida"] = df["unidad_original"].map(TIPO_MEDIDA).fillna("otro")
df["cantidad_kg"] = (df["cantidad_original"] * df["unidad_original"].map(FACTOR_A_KG)).round(9)   
df["cantidad_litros"] = df["cantidad_original"].where(df["unidad_original"] == "lt")
df["cantidad_unidades"] = df["cantidad_original"].where(df["tipo_medida"].isin(["unidades", "plantas"]))

df["cantidad_no_positiva"] = df["cantidad_original"] <= 0
df.loc[df["cantidad_no_positiva"], ["cantidad_kg", "cantidad_litros", "cantidad_unidades"]] = np.nan

registrar(12, f"Unidades: g a kg; litros y unidades aparte. tipo_medida={df.tipo_medida.value_counts().to_dict()}. "
              f"Cantidades <=0 marcadas={df.cantidad_no_positiva.sum()}",
          len(df), len(df), (df["unidad_original"] != "kg").sum())

Paso 12: Unidades: g a kg; litros y unidades aparte. tipo_medida={'masa': 624785, 'unidades': 76403, 'plantas': 11919, 'volumen': 5721, 'otro': 14}. Cantidades <=0 marcadas=19
   filas antes=718,842 | después=718,842 | afectadas=513,629


## Ciudad y puerto
- ciudad: sin tildes (las letras no latinas, como el cirílico, se respetan), sin mayúsculas raras y sin prefijos rusos tipo G. Kazan.
  No se corta por la coma: cada país escribe distinto (Ceuta, Ciudad De,  Komi Republic, Syktyvkar,  Yekaterinburg, Sverdlovsk Region)
  y cortar rompía nombres (Ceuta, Ciudad De a Ciudad De).
- es_puerto: el lugar del decomiso es: Seaport/Riverport station/Harbour.


In [47]:
ciudad = (df["ciudad_original"]
          .str.normalize("NFKD")
          .str.replace(r"(?<=[A-Za-z])[\u0300-\u036f]", "", regex=True)               
          .str.normalize("NFC")
          .str.replace(r"^(G|Gor|Г|Гор)\.\s+", "", regex=True, case=False)            
          .str.strip().str.title())
df["ciudad"] = ciudad.replace({"": "Sin dato", "Sin Dato": "Sin dato"})
df["es_puerto"] = df["lugar_fisico"].eq("Seaport/Riverport station/Harbour")

registrar(13, f"Ciudad normalizada ({df.ciudad_original.nunique():,} a {df.ciudad.nunique():,} valores). "
              f"Filas en puerto={df.es_puerto.sum():,}. Alias aún sin resolver",
          len(df), len(df), (df.ciudad != df.ciudad_original).sum())
display(df.loc[df.es_puerto].groupby("pais").size().sort_values(ascending=False).head(10).rename("filas en puerto"))

Paso 13: Ciudad normalizada (39,291 a 38,365 valores). Filas en puerto=8,886. Alias aún sin resolver
   filas antes=718,842 | después=718,842 | afectadas=51,091


pais
Spain       4998
Italy       1407
Belgium      971
Ireland      485
Sweden       367
Greece       260
Portugal     116
Finland       87
Malta         54
Germany       41
Name: filas en puerto, dtype: int64

## Posibles duplicados de contenido
Filas con distinto id pero exactamente los mismos datos (misma fecha, ciudad, droga, cantidad…).
No se borran: dos decomisos reales pueden coincidir. Se marcan para poder decidir después en Power BI.

In [48]:
CLAVE_DUPLICADO = ["fecha", "iso3", "ciudad_original", "region_administrativa", "droga_original",
                   "unidad_original", "cantidad_original", "lugar_fisico", "transporte"]

df["n_repeticiones"] = df.groupby(CLAVE_DUPLICADO)["id_decomiso"].transform("size")
df["posible_duplicado"] = df.duplicated(subset=CLAVE_DUPLICADO, keep="first")

registrar(14, "Posibles duplicados de contenido (marcados, no borrados)", len(df), len(df), df.posible_duplicado.sum())
display(df.loc[df.posible_duplicado, "pais"].value_counts().head(8).rename("posibles duplicados"))

Paso 14: Posibles duplicados de contenido (marcados, no borrados)
   filas antes=718,842 | después=718,842 | afectadas=21,429


pais
Sweden                6090
Ireland               4063
Denmark               2809
Spain                 2560
Italy                 2270
Russian Federation     832
Switzerland            685
Austria                619
Name: posibles duplicados, dtype: int64

## Cantidades extremas
Masas de más de 10.000 kg en un solo decomiso: se marcan para revisarlas (no se borran).

In [49]:
df["cantidad_extrema"] = df["cantidad_kg"] > 10_000
registrar(15, "Masas > 10.000 kg marcadas (cantidad_extrema)", len(df), len(df), df.cantidad_extrema.sum())
display(df.nlargest(5, "cantidad_kg")[["fecha", "pais", "ciudad", "droga", "cantidad_kg"]])

Paso 15: Masas > 10.000 kg marcadas (cantidad_extrema)
   filas antes=718,842 | después=718,842 | afectadas=53


,fecha,pais,ciudad,droga,cantidad_kg
682568,2025-05-07,Czechia,Praha,GBL,66400.0
92940,2013-05-03,Spain,Cordoba,Cannabis resin (hashish),52600.0
38493,2015-06-26,Spain,Sin dato,Cannabis resin (hashish),47890.0
62630,2014-06-25,Italy,Pantelleria,Cannabis resin (hashish),42672.0
93177,2013-04-29,Spain,Algeciras,Cannabis resin (hashish),32376.5


## Columnas de fecha
Para poder ver la evolución por año, trimestre y mes.

In [50]:
df["anio"] = df["fecha"].dt.year
df["mes"] = df["fecha"].dt.month
df["trimestre"] = df["fecha"].dt.quarter
df["anio_mes"] = df["fecha"].dt.strftime("%Y-%m")
registrar(16, "Columnas anio, mes, trimestre, anio_mes", len(df), len(df), len(df))

Paso 16: Columnas anio, mes, trimestre, anio_mes
   filas antes=718,842 | después=718,842 | afectadas=718,842


## Tabla limpia (Silver)
Se ordenan las columnas y se comprueba que cada decomiso tiene un id único.


In [51]:
COLUMNAS_FACT = [
    "id_decomiso", "fecha", "anio", "mes", "trimestre", "anio_mes",
    "iso3", "pais", "region", "subregion", "region_administrativa", "ciudad", "ciudad_original",
    "droga", "droga_original", "grupo_droga", "grupo_unodc_71",
    "unidad_original", "tipo_medida", "cantidad_original", "cantidad_kg", "cantidad_litros", "cantidad_unidades",
    "lugar_fisico", "lugar_disponible", "es_puerto", "transporte", "transporte_disponible",
    "posible_duplicado", "n_repeticiones", "cantidad_extrema", "cantidad_no_positiva", "archivo_origen",
]
fact_decomiso = df[COLUMNAS_FACT].sort_values(["fecha", "id_decomiso"]).reset_index(drop=True)
assert fact_decomiso["id_decomiso"].is_unique, "hay ids repetidos"
registrar(17, f"Tabla limpia lista: {len(fact_decomiso):,} filas x {len(COLUMNAS_FACT)} columnas", len(fact_decomiso), len(fact_decomiso))

Paso 17: Tabla limpia lista: 718,842 filas x 33 columnas
   filas antes=718,842 | después=718,842 | afectadas=0


## Exportar
CSV en UTF-8 con BOM (utf-8-sig) para que Excel y Power BI lean bien las tildes. 

In [52]:
fact_decomiso.to_csv(SILVER / "ids_decomisos_limpio.csv.gz", index=False, encoding="utf-8-sig",
                     date_format="%Y-%m-%d", float_format="%.9f")   

(df[["droga_original", "droga", "grupo_droga"]].drop_duplicates()
   .sort_values(["grupo_droga", "droga"])
   .to_csv(SILVER / "normalizacion_droga_ids.csv", index=False, encoding="utf-8-sig"))

equivalencia.to_csv(SILVER / "equivalencia_grupo_droga.csv", index=False, encoding="utf-8-sig")

log = pd.DataFrame(registro)
log.to_csv(SILVER / "log_ids_limpieza.csv", index=False, encoding="utf-8-sig")
display(log[["paso", "que_se_hizo", "filas_antes", "filas_despues", "filas_afectadas"]])

,paso,que_se_hizo,filas_antes,filas_despues,filas_afectadas
0,1,Carga de 7 ficheros,0,718856,7
1,2,Quitar filas de pie (vacías y 'Filtros aplicad...,718856,718842,14
2,3,Duplicados por Row Number,718842,718842,0
3,4,Renombrar columnas,718842,718842,0
4,5,"Tipos: fechas no válidas=0, cantidades no numé...",718842,718842,0
5,6,Quitar espacios sobrantes en columnas de texto,718842,718842,2210
6,7,"Nulos: {'ciudad_original': 22668, 'region_admi...",718842,718842,1094512
7,8,Un solo nombre por iso3 (ej. 'Netherlands (Kin...,718842,718842,5
8,9,Unificar subregiones. Países con más de una su...,718842,718842,172333
9,10,"Droga: 262 a 261 nombres, 13 grupos (clasifica...",718842,718842,2778


## Resumen final


In [53]:
filas_entrada = registro[0]["filas_despues"]
pies_quitados = registro[1]["filas_afectadas"]
ids_repetidos = registro[2]["filas_afectadas"]
esperadas = filas_entrada - pies_quitados - ids_repetidos
print(f"Entrada {filas_entrada:,} - pies {pies_quitados:,} - ids repetidos {ids_repetidos:,} = {esperadas:,} | salida {len(fact_decomiso):,}")
assert esperadas == len(fact_decomiso)
print("OK: no se ha perdido ninguna fila")

Entrada 718,856 - pies 14 - ids repetidos 0 = 718,842 | salida 718,842
OK: no se ha perdido ninguna fila
